# **Refactoring Functions**

### **Use functions to keep code "DRY"**

The developer who learns to **recognize duplication**, and understands how to **eliminate it through proper abstraction** (i.e. defining the right functions or methods), can produce much **cleaner code** than one who continuously uses **unnecessary repetition**.

Every line of code that goes into an application must be **maintained**, and is a potential source of future **bugs**. **Duplication** needlessly **bloats the codebase**, resulting in more opportunities for bugs and adding **accidental complexity** to the system.

The bloat that duplication adds to the system also makes it more difficult for developers working with the system to fully **understand** the entire system, or to be certain that **changes** made in one location do not also need to be made in other places that duplicate the logic they are working on.

**DRY** requires that *every piece of knowledge must have a single, unambiguous, authoritative representation within a system*.

### **Function Refactoring Flow**

```mermaid
flowchart TD
    A[Function feels hard to read] --> B{Does it do one thing?}
    B -- No --> C[Split into smaller functions]
    B -- Yes --> D{Too many parameters?}
    D -- Yes --> E[Use higher-level objects or defaults]
    D -- No --> F[Add type hints and clear names]
    E --> F
    C --> F
```


In [2]:
import hashlib

import numpy as np
import pandas as pd
from IPython.display import display
from pydantic import BaseModel
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier

### **Exercise 1: Use functions to keep code "DRY"**

Refactor this example into a function so behavior stays the same and code is not repetitive.

In [3]:
loans = pd.read_csv("data/loans.csv")
loans = loans.dropna()
X = loans.drop(["loan_id", "borrower", "target"], axis=1)
y = loans["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [5]:
# Bad example
decision_tree_model = DecisionTreeClassifier()
decision_tree_model.fit(X_train, y_train)
decision_tree_accuracy = round(decision_tree_model.score(X_test, y_test) * 100, 2)
print(decision_tree_accuracy)

random_forest_model = RandomForestClassifier(n_estimators=100)
random_forest_model.fit(X_train, y_train)
random_forest_accuracy = round(random_forest_model.score(X_test, y_test) * 100, 2)
print(random_forest_accuracy)

gaussian_model = GaussianNB()
gaussian_model.fit(X_train, y_train)
gaussian_accuracy = round(gaussian_model.score(X_test, y_test) * 100, 2)
print(gaussian_accuracy)

100.0
100.0
100.0


In [ ]:
# @TODO: Refactor duplicated model-training logic into one reusable function.
# 1) Create a helper that trains a model and returns the model + accuracy.
# 2) Keep the same behavior and printed accuracy output style.
# 3) Reuse the helper for all three models.

# Write your refactor below.
#Define a helper function to train a model and return the model and accuracy
def train_and_evaluate_model(model, X_train, y_train, X_test, y_test):
  model.fit(X_train, y_train)
  accuracy = round(model.score(X_test, y_test) * 100, 2)
  return model, accuracy

#Use the helper function to train and evaluate the models
decision_tree_model, decision_tree_accuracy = train_and_evaluate_model(
    DecisionTreeClassifier(), X_train, y_train, X_test, y_test
)
print(decision_tree_accuracy)

random_forest_model, random_forest_accuracy = train_and_evaluate_model(
    RandomForestClassifier(n_estimators=100), X_train, y_train, X_test, y_test
)
print(random_forest_accuracy)

gaussian_model, gaussian_accuracy = train_and_evaluate_model(
    GaussianNB(), X_train, y_train, X_test, y_test
)
print(gaussian_accuracy)



100.0
100.0
100.0


#### Example Solution

In [7]:
# Difference vs bad example: one helper removes duplicated model-training logic.
def train_model(model_class, X_train, y_train, X_test, y_test, **kwargs):
    model = model_class(**kwargs)
    model.fit(X_train, y_train)

    accuracy_score = round(model.score(X_test, y_test) * 100, 2)
    # Keep output style simple (numeric accuracy) while avoiding repetition.
    print(f"Accuracy ({model_class.__name__}): {accuracy_score}")

    return model, accuracy_score


decision_tree_model, decision_tree_accuracy = train_model(
    DecisionTreeClassifier, X_train, y_train, X_test, y_test
)
random_forest_model, random_forest_accuracy = train_model(
    RandomForestClassifier, X_train, y_train, X_test, y_test, n_estimators=100
)
gaussian_model, gaussian_accuracy = train_model(
    GaussianNB, X_train, y_train, X_test, y_test
)

Accuracy (DecisionTreeClassifier): 100.0
Accuracy (RandomForestClassifier): 100.0
Accuracy (GaussianNB): 100.0


> **Tip:** Notice how the symmetry of the 3 code blocks in the bad example made it easier for us to identify and refactor the duplicated code? One useful practice in eliminating duplication is to first make the duplication as obvious as possible. This makes it easier for us to identify opportunities for extracting the duplication into their appropriate homes.

### **Exercise 2: Functions should do one thing**

This is by far the most **important rule** in software engineering. When functions do more than one thing, they are harder to compose, test, and reason about. When you can isolate a function to just **one action**, they can be refactored easily and your code will read much cleaner.

Refactor this example so behavior stays the same and readability improves.

In [14]:
client_list = [
    ("John Doe", "active"),
    ("Jane Roe", "active"),
    ("Emma Bardot", "inactive"),
    ("Peter Parker", "inactive"),
]

In [12]:
# Bad example
def email_clients(clients: list):
    """Filter active clients and send them an email."""
    for client in clients:
        if client[1] == "active":
            print(f"Sending email to {client[0]}")


email_clients(client_list)

Sending email to John Doe
Sending email to Jane Roe


In [ ]:
# @TODO: Split this function so each function does one thing.
# 1) Create one function for filtering active clients.
# 2) Create one function for emailing those clients.
# 3) Call both functions in sequence.

# Write your refactor below.
# 1 Function to filter active clients
def get_active_clients(clients: list) -> list:
  """Return alist containing only active clients."""
  return [client for client in clients if client[1] == "active"]


# 2 Function to email the filtered clients
def email_clients(active_clients: list):
  """Send an email to each client in the list ."""
  for client in active_clients:
    print(f"Sending email to {client[0]}")


# 3 Call both functions in sequence;
active_clients = get_active_clients(client_list)
email_clients(active_clients)



Sending email to John Doe
Sending email to Jane Roe


#### Example Solution

In [ ]:
# Difference vs bad example: split filtering and emailing into two focused functions.
def get_active_clients(client_rows):
    return [row[0] for row in client_rows if row[1] == "active"]


def email_clients(client_names):
    for name in client_names:
        print(f"Sending email to {name}")


active_clients = get_active_clients(client_list)
email_clients(active_clients)

### **Exercise 3: Functions should only be one level of abstraction**

A **level of abstraction** refers to how much detail a piece of code shows: higher-level abstraction focuses on *what* the code does, while lower-level abstraction shows *how* it does it. When you have **more than one level of abstraction**, your function is usually **doing too much**. Splitting up functions leads to reusability and easier testing.

Refactor this example so behavior stays the same and readability improves.

In [19]:
# Bad example
def parse_better_js_alternative(code: str) -> None:
    regexes = [
        # ...
    ]

    statements = code.split()
    tokens = []
    for regex in regexes:
        for statement in statements:
            pass

    ast = []
    for token in tokens:
        # Lex.
        pass

    for node in ast:
        # Parse.
        pass


parse_better_js_alternative("let x = 5;")

In [ ]:
# @TODO: Refactor to one level of abstraction per function.
# 1) Keep `parse_better_js_alternative` as an orchestrator.
# 2) Extract tokenization into `tokenize(...)`.
# 3) Extract parsing into `parse(...)`.

# Write your refactor below.
def tokenize(code: str) -> list:
    """Tokenize the input code string."""
    regexes = []
    statements = code.split()
    tokens = []
    for regex in regexes:
        for statement in statements:
            pass  # Tokenization logic
    return tokens


def parse(tokens: list) -> None:
    """Lex tokens into an AST and parse the nodes;"""
    ast = []
    for token in tokens:
        # lexing logic
        pass

    for node in ast:
        # Parsing logic
        pass


def parse_better_js_alternative(code: str) -> None:
    """Orchestrate the parsing pipeline at a high level of abstraction."""
    tokens = tokenize(code)
    parse(tokens)


parse_better_js_alternative("let x = 5;")




#### Example Solution

In [ ]:
# Difference vs bad example: orchestrator delegates lower-level work to helpers.
def parse_better_js_alternative(code: str) -> None:
    tokens = tokenize(code)
    syntax_tree = parse(tokens)

    for node in syntax_tree:
        # Parse.
        pass


def tokenize(code: str) -> list:
    regexes = [
        # ...
    ]

    statements = code.split()
    tokens = []
    for regex in regexes:
        for statement in statements:
            # Append the statement to tokens.
            pass

    return tokens


def parse(tokens: list) -> list:
    syntax_tree = []
    for token in tokens:
        # Append the parsed token to the syntax tree.
        pass

    return syntax_tree


parse_better_js_alternative("let x = 5;")

### **Exercise 4: Function names should say what they do**

Refactor this example so behavior stays the same and functionality is clearer.

In [22]:
# Bad example
def handle_client(client: str) -> None:
    # Function to send out emails ...
    print(f"Email sent to {client}")

In [21]:
# @TODO: Rename this function so the name clearly states the action.
# 1) Keep the same parameter and behavior.
# 2) Use a name that reflects sending an email.

# Write your refactor below.
def send_email_to_client(client: str) -> None:
    """Send an email to the specified client."""
    print(f"Email sent to {client}")


#### Example Solution

In [ ]:
# Difference vs bad example: function name now states the action explicitly.
def send_email(client: str) -> None:
    # Function to send out emails ...
    print(f"Email sent to {client}")

### **Use type hints to improve readability**

Using **type hints** can make your code more readable and maintainable. Your development experience will also improve because your IDE will be able to give you better **auto-complete suggestions** about function/method names and parameters.

> **Note:** Type hints are actually meant to be entirely **ignored by the Python runtime**! But you can use tools like [**mypy**](https://www.mypy-lang.org/), or activate **type checking** in [**VSCode**](https://www.emmanuelgautier.com/blog/enable-vscode-python-type-checking).

Without type hints, we are forced to embed type information in variable names (e.g. `pd_series`). This can make variable names unnecessarily long. Also, our IDE is not able to give us auto-complete hints and, as a result, we have to hop to the source file to find out what parameters this function accepts.


In [ ]:
# Bad example
def categorize_continuous_values(pd_series, num_bins):
    bins = pd.cut(pd_series, num_bins, retbins=True)[1]
    return pd.Series(np.digitize(pd_series, bins, right=True))


sample_values = pd.Series([0.2, 1.4, 2.1, 3.8])
categorize_continuous_values(sample_values, num_bins=3)

With type hints we can name our variables sensibly, and the IDE now offers better autocompletion, making us more productive and less error-prone. Note that if you **hover over the function**, you now see the data types we assigned to our variables.

In [ ]:
def categorize_continuous_values(values: pd.Series, num_bins: int) -> pd.Series:
    bins = pd.cut(values, num_bins, retbins=True)[1]
    return pd.Series(np.digitize(values, bins, right=True))


sample_values: pd.Series = pd.Series([0.2, 1.4, 2.1, 3.8])
categorize_continuous_values(sample_values, num_bins=3)

### **Exercise 5: Avoid side effects**

A function produces a **side effect** if it does anything other than take a value in and return another value or values.

Refactor this example so behavior stays the same and the code doesn't break.

In [24]:
# Bad example
# Global variable referenced by following function.
# If another function used this name, now it'd be an array and could break.
name = "Ryan McDermott"


def split_into_first_and_last_name() -> None:
    global name
    if not isinstance(name, str):
        raise TypeError("name is no longer a string")
    name = name.split()


split_into_first_and_last_name()
print(name)  # ['Ryan', 'McDermott']

# Calling this function the second time would throw a
# TypeError since `name` would now be a list, not a string.

try:
    split_into_first_and_last_name()
except TypeError as type_error:
    print(f"Error: {type_error}")

['Ryan', 'McDermott']
Error: name is no longer a string


In [29]:
# @TODO: Remove global mutation and make this function side-effect free.
# 1) Pass `name` as a function argument.
# 2) Return the split result instead of mutating global state.
# 3) Show both the original and new value outputs.

# Write your refactor below.
def split_into_first_and_last_name(name: str) -> list:
    return name.split()
full_name = "Shiar Osman"
first_name, last_name = split_into_first_and_last_name(full_name)
# Sort First and last name in array and print them
first_name, last_name = sorted([first_name, last_name])
print(f"First name: {first_name}")
print(f"Last name: {last_name}")
print(f"Original full name: {full_name}")
#print the array
print(f"Sorted names: {[first_name, last_name]}")



First name: Osman
Last name: Shiar
Original full name: Shiar Osman
Sorted names: ['Osman', 'Shiar']


#### Example Solution

In [27]:
# Difference vs bad example: no global mutation; function is pure and returns a value.
def split_into_first_and_last_name(name: str) -> list[str]:
    return name.split()


full_name = "Ryan McDermott"
new_name = split_into_first_and_last_name(full_name)

print(full_name)  # 'Ryan McDermott'
print(new_name)  # ['Ryan', 'McDermott']

Ryan McDermott
['Ryan', 'McDermott']


### **Exercise 6: Avoid unexpected side effects on values passed as function parameters**

We can unexpectedly change the values passed to our functions, even though our functions appear to be pure. This commonly happens with **mutable objects** (e.g. lists, dictionaries, instances of classes, pandas DataFrames) because **function parameters hold references to those objects**.

Refactor this example so behavior stays the same and there are no unexpected side effects.

In [30]:
original = pd.DataFrame(
    {
        "values": [1, 2, 3],
    }
)


def multiply_column_by_10(df, column_name):
    df["multiplied column"] = df[column_name] * 10

    return df


new = multiply_column_by_10(original, "values")
original.head()  # Original DataFrame is mutated and has now 2 columns

,values,multiplied column
0,1,10
1,2,20
2,3,30


In [33]:
# @TODO: Avoid mutating the input DataFrame.
# 1) Keep the function behavior, but return a safe copy.
# 2) Ensure `original` stays unchanged after calling the function.

# Write your refactor below.

original = pd.DataFrame(
    {
        "values": [1, 2, 3],
    }
)
def add_one_safe(df: pd.DataFrame) -> pd.DataFrame:
    """Return a new DataFrame with a new column that is the original column multiplied by 10."""
    new_df = df.copy()
    new_df["multiplied column"] = new_df["values"] * 10
    return new_df

new = add_one_safe(original)
print("Original DataFrame:")
print(original)
print("New DataFrame:")
print(new)

Original DataFrame:
   values
0       1
1       2
2       3
New DataFrame:
   values  multiplied column
0       1                 10
1       2                 20
2       3                 30


#### Example Solution

In [32]:
# Difference vs bad example: operate on a copy so the input DataFrame stays unchanged.
original = pd.DataFrame(
    {
        "values": [1, 2, 3],
    }
)


def multiply_column_by_10(df, column_name):
    safe_copy = df.copy()
    safe_copy["multiplied column"] = safe_copy[column_name] * 10

    return safe_copy


new = multiply_column_by_10(original, "values")
original.head()  # Original DataFrame is unchanged

,values
0,1
1,2
2,3


Copying the DataFrame makes this approach safer because the original object remains unchanged. However, it is not the only way to make this specific method safe, and this extra safety comes with a performance cost, which may become significant for large DataFrames. 

### **Exercise 7: Use default arguments instead of short-circuiting or conditionals**

A **slug** is a unique identifier generated from a string using a hash function. Here we are using the `hashlib` library to create a slug for a brewery, generated from its name (and if the name is not provided, we want to use a default one).

We can use **default arguments** in functions to provide a fallback value when no input is given, instead of relying on conditional statements or short-circuiting.

Refactor this example so behavior stays the same and readability improves.


In [ ]:
# Bad example
def create_micro_brewery(name):
    name = "Hipster Brew Co." if name is None else name
    slug = hashlib.sha1(name.encode()).hexdigest()
    return f"Microbrewery(name='{name}', slug='{slug}')"


create_micro_brewery(None)  # 'Hipster Brew Co.' is used as default name

In [ ]:
# @TODO: Use a default argument in the function signature.
# 1) Remove the conditional assignment inside the function.
# 2) Keep behavior equivalent.

# Write your refactor below.


#### Example Solution

In [ ]:
# Difference vs bad example: use a default parameter instead of conditional reassignment.
def create_micro_brewery(name: str = "Hipster Brew Co."):
    slug = hashlib.sha1(name.encode()).hexdigest()
    return f"Microbrewery(name='{name}', slug='{slug}')"


create_micro_brewery()  # Uses default name

### **Exercise 8: Don't use flags as function parameters**

A **flag** is a binary indicator variable that takes only two values (typically 0/1 or True/False). Flags tell your user that this function does more than one thing. **Functions should do one thing**. Split your functions if they are following different code paths based on a boolean.

Refactor this example so behavior stays the same and no boolean is needed.

In [ ]:
# Bad example
def read_csv_file(name: str, head: bool) -> None:
    if head:
        display(pd.read_csv(name).head())  # Prints first few lines
    else:
        display(pd.read_csv(name))  # Prints entire file


read_csv_file("data/loans.csv", head=True)  # Prints first few lines

In [ ]:
# @TODO: Split behavior into separate functions instead of using a flag.
# 1) Keep one function for printing the head.
# 2) Create another one for printing the entire file.

# Write your refactor below.


#### Example Solution

In [ ]:
# Difference vs bad example: split two behaviors into two functions (no boolean flag).
def read_csv_file(name: str) -> None:
    display(pd.read_csv(name))  # Prints entire file


def read_csv_file_head(name: str) -> None:
    display(pd.read_csv(name).head())  # Prints first few lines


read_csv_file_head("data/loans.csv")

### **Exercise 9: Function arguments should ideally be 2 or less**

**Limiting the amount of function parameters** is incredibly important because it makes testing your function easier. Having more than three leads to a **combinatorial explosion** where you have to test tons of different cases with each separate argument.

One or two arguments is ok; three should be avoided. Anything more than that should be consolidated (usually a higher-level object will suffice).

Refactor this example so behavior stays the same and the amount of arguments is reduced.

In [34]:
# Bad example
def create_banner(title, subtitle, button_text, color, dismissible):
    return {
        "title": title,
        "subtitle": subtitle,
        "button_text": button_text,
        "color": color,
        "dismissible": dismissible,
    }


banner = create_banner("Welcome", "Try our features", "Start", "blue", True)
print(banner)

{'title': 'Welcome', 'subtitle': 'Try our features', 'button_text': 'Start', 'color': 'blue', 'dismissible': True}


In [35]:
# @TODO: Refactor this function to avoid too many parameters.
# 1) Create a config object (dict or Pydantic model).
# 2) Pass that object into a smaller function signature.

# Write your refactor below.
def create_banner(config: dict) -> dict:

    return {
        "title": config.get("title"),
        "subtitle": config.get("subtitle"),
        "button_text": config.get("button_text"),
        "color": config.get("color"),
        "dismissible": config.get("dismissible"),
    }
config = {
    "title": "Welcome",
    "subtitle": "Try our features",
    "button_text": "Start",
    "color": "blue",
    "dismissible": True,
}
banner = create_banner(config)
print(banner)



{'title': 'Welcome', 'subtitle': 'Try our features', 'button_text': 'Start', 'color': 'blue', 'dismissible': True}


#### Example Solution

In [36]:
# Difference vs bad example: pass one config object instead of many parameters.
def create_banner(config: dict) -> dict:
    return {
        "title": config["title"],
        "subtitle": config["subtitle"],
        "button_text": config["button_text"],
        "color": config["color"],
        "dismissible": config["dismissible"],
    }


banner_config = {
    "title": "Welcome",
    "subtitle": "Try our features",
    "button_text": "Start",
    "color": "blue",
    "dismissible": True,
}

banner = create_banner(banner_config)
print(banner)

{'title': 'Welcome', 'subtitle': 'Try our features', 'button_text': 'Start', 'color': 'blue', 'dismissible': True}


> **Note:** The following examples use **Object-Oriented Programming** and **Pydantic**. Both topics will be introduced in the following notebooks of this repository.

Let's see another example:


In [ ]:
def create_menu(config: dict) -> dict:
    return {
        "title": config["title"],
        "body": config["body"],
        "button_text": config["button_text"],
        "cancellable": config["cancellable"],
    }


menu_config = {
    "title": "My Menu",
    "body": "Something about my menu",
    "button_text": "OK",
    "cancellable": False,
}

menu = create_menu(menu_config)
print(menu)

Refactoring using **OOP**:

In [ ]:
class Menu:
    def __init__(self, config: dict):
        self.title = config["title"]
        self.body = config["body"]
        self.button_text = config["button_text"]
        self.cancellable = config["cancellable"]


menu = Menu(
    {
        "title": "My Menu",
        "body": "Something about my menu",
        "button_text": "OK",
        "cancellable": False,
    }
)

menu.title

Or a little bit more fancy with **Pydantic**:

In [ ]:
class Menu(BaseModel):
    title: str
    body: str
    button_text: str
    cancellable: bool


menu_dict = {
    "title": "My Menu",
    "body": "Something about my menu",
    "button_text": "OK",
    "cancellable": False,
}

menu = Menu.model_validate(menu_dict)
print(menu)